In [1]:
## General imports
import pickle
import numpy as np
from multiprocessing import Manager, Pool, cpu_count
from functools import partial
from shared_memory_dict import SharedMemoryDict
## Local imports
import get_data_matrix as getMat
import mapLearningUtils as mlu
import multiprocessingUtils as mpu
import train_MappingJM as trMap

In [2]:
## Set configuration
# Data handling
src_data       = ['CALIB', 'ASSIST']
data_type_list = ['exoPositions', 'exoVelocities', 'humanPositions', 'humanVelocities', 'wristSensor']
list_vars      = ['q3', 'q4', 'qd3', 'qd4', 'Fx', 'Fy', 'Fz', 'Mx', 'My', 'Mz', 'l_a', 'l_fa',
                  'shoulder_elv', 'elbow_flexion', 'dshoulder_elv', 'delbow_flexion', 'trial', 'subj']
# CALIB params
path_calib         = './data_CALIB'
name_expe          = 'CALIB'
save_path_calib    = './data_CALIB/dataMatCalib.pkl'
nb_subj            = 17
cond               = 'MJ'
nb_trials_calib    = 10
duration_idx_calib = 2900
params_calib       = {'path': path_calib, 'expeName': name_expe, 'save_path': save_path_calib, 'nb_subj': nb_subj, 'condition': cond,
                      'nb_trials': nb_trials_calib, 'dataTypes': data_type_list, 'listOfVariables': list_vars, 'durationIndex': duration_idx_calib}
# ASSIST params
path_assist         = './data_ASSIST'
name_expe           = 'ASSIST'
save_path_assist    = './data_ASSIST/dataMatAssist.pkl'
subjs_list          = ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S09', 'S10', 'S12', 'S14']
conds_list          = ['T', 'ES', 'EG']
phases_list         = ['PE', 'PS', 'RE', 'RS']
nb_trials_calib     = 26
duration_idx_assist = 500
params_assist       = {'path': path_assist, 'expeName': name_expe, 'save_path': save_path_assist, 'subjList': subjs_list, 'conditions': conds_list,
                      'nb_trials': nb_trials_calib, 'phases': phases_list, 'dataTypes': data_type_list, 'listOfVariables': list_vars,
                      'durationIndex': duration_idx_assist}

In [3]:
## Get data matrices
# Import CALIB data
data_calib = getMat.get_data(src_data[0], params_calib)
print('CALIB data shape: ' + str(data_calib.shape))
# Import ASSIST data
data_assist = getMat.get_data(src_data[1], params_assist)
print('ASSIST data shape: ' + str(data_assist.shape))

Loaded calibration data from pickle.
CALIB data shape: (493000, 18)
Loaded assistance data from pickle.
ASSIST data shape: (227322, 18)


In [4]:
## Pre-processing parameters
savePath      = './all_params/folded_ablated_data.pkl'
ablationsList = ['velocity', 'anthropo', 'forcestorques', 'forces', 'torques', 'Fx', 'Fy', 'Fz', 'Tx', 'Ty', 'Tz']
foldsList     = ['2', '5']
forceLoc      = 'wrist'

params_prepro = {'ablations': ablationsList, 'folds': foldsList, 'forceLoc': forceLoc, 'savePathPrepro': savePath}

In [5]:
## Prepare dictionnary with all ablations and foldings
fold_dict_all = mlu.get_all_folded_ablated_data(data_calib, data_assist, params_prepro)

Loaded folded and ablated data from pickle.


In [ ]:
## Mapping parameters
# Common
fitTypesList = ['MVLR', 'MVPR'] # FNO, LSTM

# MVPR
degreesList = [2, 3, 4, 5, 10] # Degree of the fitted polynomial for each input

# FNO
nbModesList   = [4, 8, 16, 32]       # Number of modes [number of low Fourier frequencies]
nbHChanList   = [8, 16, 32, 64] # Number of hidden channels [dimensionnality of the network's internal features]
nbMaxIterList = [100, 250, 500]      # Maximum number of training iterations
limLoss       = 1e-6                 # Convergence limit (if less improvement between two iterations then training stops)

# LSTM
# TODO

In [ ]:
## Prepare lists of parameters dictionnaries to train different versions of the models
list_params_sets = []
for fitting in fitTypesList:
    for ablation in ablationsList:
        for fold in foldsList:
            fold_name = fold + '-fold'
            if fitting == 'MVLR':
                dict_params = {'model': fitting, 'ablation': ablation, 'fold': fold_name, 'nb_folds': int(fold)}
                list_params_sets.append(dict_params)
            elif fitting == 'MVPR':
                for degree in degreesList:
                    dict_params = {'model': fitting, 'degree': degree, 'ablation': ablation, 'fold': fold_name, 'nb_folds': int(fold)}
                    list_params_sets.append(dict_params)
            elif fitting == 'FNO':
                "FNO is not yet handled: Needs HPC"
            else:
                "The requested fitting type is not yet handled."

In [8]:
## Shared dictionnary parameters, creation and filling
# Global variable defined in initializer
_global_shared_dict = None

# Dictionnary name and size
shared_dict_name = 'SHM_foldedAblated_data'
shared_dict_size = int(len(pickle.dumps(fold_dict_all)) * 1.1)

# Dictionnary initialization and filling by copy of the folded and ablated data dictionnary
fold_dict_all_keys = list(fold_dict_all.keys())
shared_dict = SharedMemoryDict(name = shared_dict_name, size = shared_dict_size)
for key in fold_dict_all_keys:
    value = fold_dict_all[key]
    shared_dict[key] = value
    del fold_dict_all[key]

fold_dict_all = {}

In [9]:
## Run training of models in parallel
# Create multiprocessing pool
nb_processes = cpu_count() - 1
pool = Pool(processes = nb_processes, initializer = mpu.worker_init, initargs = (shared_dict_name, shared_dict_size))
# Run multiprocessed traning
list_trainedModels = pool.map(mpu.run_training, list_params_sets)

AttributeError: module 'multiprocessingUtils' has no attribute 'run_training'